# 📰 LSTM을 활용한 로이터 뉴스 분류기

## Reuters-11228 데이터셋
- **출처**: Reuters 통신사 뉴스 기사 (1987년)
- **규모**: 11,228개 기사
- **클래스**: 46개 카테고리 (경제, 정치, 스포츠 등)
- **특징**: 클래스 불균형 심함 (상위 10개 클래스가 대부분을 차지)

## LSTM 구조

```
Input (정수 시퀀스)
     ↓
Embedding(10000, 128)   ← 단어 → 밀집 벡터
     ↓
LSTM(128→256, layers=2, dropout=0.3)
     ↓
Global Average Pooling  ← 전체 시퀀스 평균
     ↓
Dense(256→128) → ReLU → Dropout(0.4)
     ↓
Dense(128→46) → Softmax  ← 46개 클래스
```

## 다중 분류 vs 이진 분류

| | 이진 분류 | 다중 분류 |
|--|---------|----------|
| **출력층** | Sigmoid (1개) | Softmax (N개) |
| **손실함수** | BCELoss | CrossEntropyLoss |
| **평가 지표** | Accuracy, AUC | Accuracy, Macro/Micro F1 |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

## 1️⃣ 로이터 데이터셋 로드 (Keras 내장)

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB = 10000
MAX_LEN   = 200
NUM_CLASSES = 46

(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=MAX_VOCAB)

print(f'Train: {len(x_train)}개  /  Test: {len(x_test)}개')
print(f'클래스 수: {len(set(y_train))} (0~{max(y_train)})')

# 클래스 분포
from collections import Counter
train_dist = Counter(y_train.tolist())
print(f'\n상위 10개 클래스 빈도:')
for cls, cnt in sorted(train_dist.items(), key=lambda x: -x[1])[:10]:
    print(f'  클래스 {cls:>2}: {cnt}개  ({cnt/len(y_train)*100:.1f}%)')

In [ ]:
# 클래스 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 전체 클래스 분포 (상위 20)
top20 = sorted(train_dist.items(), key=lambda x: -x[1])[:20]
cls_ids, cls_cnts = zip(*top20)
axes[0].bar(range(20), cls_cnts, color='#2196F3', alpha=0.8)
axes[0].set_xticks(range(20)); axes[0].set_xticklabels(cls_ids)
axes[0].set_title('상위 20개 클래스 분포 (Train)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('클래스 ID'); axes[0].set_ylabel('샘플 수')
axes[0].grid(axis='y', alpha=0.3)

# 기사 길이 분포
lengths = [len(x) for x in x_train]
axes[1].hist(lengths, bins=50, color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].axvline(MAX_LEN, color='red', linestyle='--', linewidth=2, label=f'MAX_LEN={MAX_LEN}')
axes[1].axvline(np.mean(lengths), color='orange', linestyle='--', label=f'평균={np.mean(lengths):.0f}')
axes[1].set_title('기사 길이 분포', fontsize=12, fontweight='bold')
axes[1].set_xlabel('토큰 수'); axes[1].set_ylabel('빈도')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f'길이 통계: 평균={np.mean(lengths):.0f} / 중앙값={np.median(lengths):.0f} / 최대={np.max(lengths)}')
print(f'MAX_LEN={MAX_LEN} 이하 비율: {np.mean([l<=MAX_LEN for l in lengths])*100:.1f}%')

## 2️⃣ 전처리 & 데이터로더

In [ ]:
# 패딩 (뒤쪽 — 뉴스의 앞부분이 중요)
x_train_pad = pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'패딩 후: {x_train_pad.shape}')


class ReutersDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


BATCH_SIZE = 64

# 학습/검증 분리
val_size = int(len(x_train_pad) * 0.15)
train_ds = ReutersDataset(x_train_pad[val_size:], y_train[val_size:])
valid_ds = ReutersDataset(x_train_pad[:val_size], y_train[:val_size])
test_ds  = ReutersDataset(x_test_pad, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'Train: {len(train_ds)} / Valid: {len(valid_ds)} / Test: {len(test_ds)}')

## 3️⃣ LSTM 뉴스 분류 모델

### 풀링 전략: 마지막 Hidden vs Global Average

- **Last Hidden**: 마지막 시점 정보 → 뉴스 앞부분 소실 위험
- **Global Avg Pool**: 전체 시퀀스 평균 → 균형 잡힌 표현 ✅
- **Max Pool**: 가장 강한 특징 → 핵심 단어 포착

In [ ]:
class LSTMNewsClassifier(nn.Module):
    def __init__(self,
                 vocab_size    = 10000,
                 embed_dim     = 128,
                 hidden_dim    = 256,
                 num_layers    = 2,
                 num_classes   = 46,
                 dropout       = 0.3,
                 bidirectional = True,
                 pooling       = 'avg',   # 'last' / 'avg' / 'max'
                 pad_idx       = 0):
        super().__init__()
        assert pooling in ('last', 'avg', 'max')
        self.pooling       = pooling
        self.bidirectional = bidirectional
        self.num_layers    = num_layers
        self.hidden_dim    = hidden_dim

        # ① Embedding
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_drop = nn.Dropout(dropout)

        # ② LSTM
        self.lstm = nn.LSTM(
            input_size    = embed_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidirectional,
            dropout       = dropout if num_layers > 1 else 0.0
        )

        # ③ 분류기
        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(lstm_out_dim),
            nn.Linear(lstm_out_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout + 0.1),
            nn.Linear(256, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.05, 0.05)
        self.embedding.weight.data[0].zero_()
        for name, p in self.lstm.named_parameters():
            if 'weight_ih' in name: nn.init.xavier_uniform_(p)
            elif 'weight_hh' in name: nn.init.orthogonal_(p)
            elif 'bias' in name: nn.init.zeros_(p)

    def forward(self, x):
        emb = self.embed_drop(self.embedding(x))    # (B, L, E)
        out, (h_n, _) = self.lstm(emb)              # out: (B, L, 2H)

        if self.pooling == 'last':
            if self.bidirectional:
                rep = torch.cat([h_n[-2], h_n[-1]], dim=1)
            else:
                rep = h_n[-1]
        elif self.pooling == 'avg':
            # PAD 토큰(0) 제외 평균 — 더 정확한 표현
            mask = (x != 0).unsqueeze(-1).float()    # (B, L, 1)
            rep  = (out * mask).sum(1) / mask.sum(1) # (B, 2H)
        elif self.pooling == 'max':
            rep = out.max(dim=1).values              # (B, 2H)

        return self.classifier(rep)                 # (B, 46)


model = LSTMNewsClassifier(
    vocab_size    = MAX_VOCAB,
    embed_dim     = 128,
    hidden_dim    = 256,
    num_layers    = 2,
    num_classes   = NUM_CLASSES,
    dropout       = 0.3,
    bidirectional = True,
    pooling       = 'avg',
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'파라미터 수: {total_params:,}')
print(model)

## 4️⃣ 클래스 불균형 처리 — 가중 손실 함수

In [ ]:
# 클래스 불균형 대응: 희귀 클래스에 더 높은 가중치
class_counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(float)
class_counts = np.where(class_counts == 0, 1, class_counts)  # 0 방지
class_weights = torch.tensor(
    len(y_train) / (NUM_CLASSES * class_counts),
    dtype=torch.float32
).to(device)

print('클래스 가중치 (상위 5개, 하위 5개):')
sorted_idx = class_weights.cpu().numpy().argsort()
print('  낮은 가중치 (빈도 높음):',
      [(i, f'{class_weights[i].item():.3f}') for i in sorted_idx[:5]])
print('  높은 가중치 (빈도 낮음):',
      [(i, f'{class_weights[i].item():.3f}') for i in sorted_idx[-5:]])

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

print('\n학습 준비 완료!')

## 5️⃣ 학습 루프

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += y.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.cpu().tolist())
    return total_loss/total, correct/total, all_preds, all_labels


EPOCHS = 15
best_val_acc = 0.0
history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>9} | {'Val Acc':>8}")
print('=' * 58)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc, _, _ = evaluate(model, valid_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    mark = ' ★' if vl_acc > best_val_acc else ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), '/tmp/best_lstm_reuters.pt')

    print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc*100:>8.2f}% | {vl_loss:>9.4f} | {vl_acc*100:>7.2f}%{mark}")

print(f'\n최고 Validation Accuracy: {best_val_acc*100:.2f}%')

## 6️⃣ 학습 곡선 & 최종 평가

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep_range = range(1, EPOCHS + 1)

axes[0].plot(ep_range, history['train_loss'], 'b-o', ms=4, label='Train')
axes[0].plot(ep_range, history['val_loss'],   'r-o', ms=4, label='Valid')
axes[0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep_range, [a*100 for a in history['train_acc']], 'b-o', ms=4, label='Train')
axes[1].plot(ep_range, [a*100 for a in history['val_acc']],   'r-o', ms=4, label='Valid')
axes[1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Reuters LSTM 학습 곡선', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('lstm_reuters_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 최고 모델로 테스트 평가
model.load_state_dict(torch.load('/tmp/best_lstm_reuters.pt', map_location=device))
te_loss, te_acc, all_preds, all_labels = evaluate(model, test_loader, criterion)

print('=' * 45)
print(f'  Test Loss    : {te_loss:.4f}')
print(f'  Test Accuracy: {te_acc*100:.2f}%')
print('=' * 45)

# 상위 10개 클래스에 대한 리포트
top10_classes = [cls for cls, _ in sorted(train_dist.items(), key=lambda x: -x[1])[:10]]

# top10 마스크
mask = [l in top10_classes for l in all_labels]
top10_preds  = [p for p, m in zip(all_preds, mask) if m]
top10_labels = [l for l, m in zip(all_labels, mask) if m]

print(f'\n[Top-10 클래스 Classification Report]')
print(classification_report(
    top10_labels, top10_preds,
    labels=top10_classes,
    target_names=[f'Class {c}' for c in top10_classes]
))

## 7️⃣ 혼동 행렬 (상위 10개 클래스)

In [ ]:
# 상위 10개 클래스 혼동 행렬
cm = confusion_matrix(top10_labels, top10_preds, labels=top10_classes)

fig, ax = plt.subplots(figsize=(12, 9))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'C{c}' for c in top10_classes],
            yticklabels=[f'C{c}' for c in top10_classes],
            ax=ax, linewidths=0.5)
ax.set_title('혼동 행렬 — 정규화 (Top-10 클래스)', fontsize=13, fontweight='bold')
ax.set_xlabel('예측 클래스'); ax.set_ylabel('실제 클래스')
plt.tight_layout(); plt.show()

## 8️⃣ 풀링 전략 비교 실험

In [ ]:
print('[풀링 전략 비교]\n')
pooling_results = {}

for pooling in ['last', 'avg', 'max']:
    m = LSTMNewsClassifier(
        vocab_size=MAX_VOCAB, embed_dim=128, hidden_dim=128,
        num_layers=1, num_classes=NUM_CLASSES, dropout=0.3,
        bidirectional=True, pooling=pooling
    ).to(device)
    opt  = optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    best = 0.0

    for epoch in range(5):
        train_epoch(m, train_loader, opt, crit)
        _, vl_acc, _, _ = evaluate(m, valid_loader, crit)
        best = max(best, vl_acc)

    pooling_results[pooling] = best
    bar = '█' * int(best * 40)
    print(f'  pooling={pooling:<4}: Val Acc = {best*100:.2f}%  |{bar}')

best_p = max(pooling_results, key=pooling_results.get)
print(f'\n★ 최고 풀링 전략: {best_p} ({pooling_results[best_p]*100:.2f}%)')

## 9️⃣ 새 뉴스 기사 분류 예측

In [ ]:
# 로이터 단어 인덱스 불러오기
word_idx = reuters.get_word_index()

@torch.no_grad()
def predict_news(text, model, word_idx, max_len=200, topk=3):
    model.eval()
    import re
    words = re.sub(r'[^a-zA-Z\s]', '', text.lower()).split()
    ids   = [word_idx.get(w, 1) + 3 for w in words]  # +3: special token offset
    ids   = ids[:max_len]
    ids   = ids + [0] * (max_len - len(ids))           # 패딩

    x      = torch.tensor([ids], dtype=torch.long, device=device)
    logits = model(x)
    probs  = torch.softmax(logits, dim=1)[0].cpu()

    top_probs, top_cls = probs.topk(topk)
    return list(zip(top_cls.tolist(), top_probs.tolist()))


# 로이터 카테고리 이름 (주요 클래스)
CATEGORY_NAMES = {
    3: 'earn', 4: 'acq', 8: 'money-fx', 10: 'crude',
    11: 'grain', 16: 'trade', 19: 'interest', 20: 'wheat',
    22: 'ship', 23: 'corn', 36: 'gold', 39: 'sugar'
}

test_articles = [
    "The company reported record quarterly earnings with profits up 45 percent driven by strong sales growth",
    "Oil prices surged to highest level as OPEC announced production cuts amid supply concerns",
    "Federal Reserve raised interest rates by 25 basis points to combat rising inflation",
    "Major acquisition deal announced as tech giant acquires startup for 2 billion dollars",
    "Wheat and corn futures rose sharply on drought concerns affecting global grain production",
    "Dollar fell against yen and euro as trade deficit figures disappointed investors",
]

model.load_state_dict(torch.load('/tmp/best_lstm_reuters.pt', map_location=device))

print('[새 뉴스 기사 분류 결과]\n')
for article in test_articles:
    results_pred = predict_news(article, model, word_idx, topk=3)
    print(f'기사: "{article[:70]}"')
    print('Top-3 예측:')
    for rank, (cls_id, prob) in enumerate(results_pred, 1):
        name = CATEGORY_NAMES.get(cls_id, f'Class-{cls_id}')
        bar  = '█' * int(prob * 30)
        print(f'  {rank}. {name:>12} (Class {cls_id:>2}): {prob:.4f} | {bar}')
    print()

## 🔟 Keras LSTM 버전 (간결한 대안)

In [ ]:
# Keras로 동일한 모델 구현 (비교용)
import tensorflow as tf
from tensorflow import keras

keras_model = keras.Sequential([
    keras.layers.Embedding(MAX_VOCAB, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=True, dropout=0.3)),
    keras.layers.Bidirectional(keras.layers.LSTM(64, dropout=0.3)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

keras_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

keras_model.summary()

# 학습 (5 epoch만)
history_keras = keras_model.fit(
    x_train_pad[val_size:], y_train[val_size:],
    validation_data=(x_train_pad[:val_size], y_train[:val_size]),
    batch_size=64,
    epochs=5,
    verbose=1
)

# 테스트 평가
keras_loss, keras_acc = keras_model.evaluate(x_test_pad, y_test, verbose=0)
print(f'\nKeras LSTM Test Accuracy: {keras_acc*100:.2f}%')